In [1]:
import os
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

os.chdir('/Users/jacksonsharpe/QSS20_Final_Project_Sharpe/data')
print(f"Working directory: {os.getcwd()}")

Working directory: /Users/jacksonsharpe/QSS20_Final_Project_Sharpe/data


In [2]:
def load_ess_full(filepath='ESSQSS20.csv'):
    """
    Same cleaning as 01_merge but loads individual-level data
    for regression analysis. Rounds 1-10 only.
    """
    cols = [
        'cntry', 'essround', 'imbgeco', 'imueclt', 'imwbcnt',
        'brncntr', 'eisced', 'eduyrs', 'agea', 'gndr',
        'uempla', 'lrscale', 'stfeco'
    ]
    ess = pd.read_csv(filepath, usecols=cols)
    
    for col in ['imbgeco', 'imueclt', 'imwbcnt']:
        ess[col] = ess[col].where(ess[col] <= 10)
    
    ess = ess[ess['brncntr'] == 1].copy()
    ess['immig_attitude'] = ess[['imbgeco', 'imueclt', 'imwbcnt']].mean(axis=1)
    ess = ess.dropna(subset=['immig_attitude'])
    
    ess['eisced'] = ess['eisced'].where(ess['eisced'] <= 7)
    ess['lrscale'] = ess['lrscale'].where(ess['lrscale'] <= 10)
    ess['agea'] = ess['agea'].where(ess['agea'] <= 120)
    ess['eduyrs'] = ess['eduyrs'].where(ess['eduyrs'] <= 30)
    ess['stfeco'] = ess['stfeco'].where(ess['stfeco'] <= 10)
    
    round_to_year = {
        1:2002, 2:2004, 3:2006, 4:2008, 5:2010,
        6:2012, 7:2014, 8:2016, 9:2018, 10:2020
    }
    ess['year'] = ess['essround'].map(round_to_year)
    ess = ess.dropna(subset=['year'])
    ess['year'] = ess['year'].astype(int)
    ess['econ_dissatisfied'] = (ess['stfeco'] <= 4).astype(int)
    
    return ess

# Load individual-level ESS
ess_full = load_ess_full()
print(f"ESS loaded: {ess_full.shape}")

policy_merge = pd.read_csv('merged_attitudes_policy.csv')[
    ['cntry', 'year', 'net_restrict_score', 'n_policy_actions', 'net_major_score', 'net_nonmajor_score']
]
ess_merged = pd.merge(ess_full, policy_merge, on=['cntry', 'year'], how='inner')
ess_merged['econ_dissatisfied'] = (ess_merged['stfeco'] <= 4).astype(int)

print(f"Before merge: {len(ess_full):,} rows")
print(f"After merge: {len(ess_merged):,} rows")
print(f"Countries: {ess_merged['cntry'].nunique()}")
print(f"net_major_score missing: {ess_merged['net_major_score'].isna().sum()}")

ESS loaded: (436194, 16)
Before merge: 436,194 rows
After merge: 392,865 rows
Countries: 30
net_major_score missing: 0


In [3]:
# Country-year level descriptives
country_year = pd.read_csv('merged_attitudes_policy.csv')

print(" Descriptive Statistics: Country-Year Level (N=233) ")
print(country_year[['mean_attitude', 'net_restrict_score', 'n_policy_actions', 'n_respondents']].describe().round(2))

# Individual level descriptives
reg_df = ess_merged.dropna(subset=['immig_attitude', 'net_restrict_score', 'agea', 'eduyrs', 'stfeco']).copy()
reg_df['net_restrict_std'] = (reg_df['net_restrict_score'] - reg_df['net_restrict_score'].mean()) / reg_df['net_restrict_score'].std()

print(f"\n Descriptive Statistics: Individual Level (N={len(reg_df):,})")
print(reg_df[['immig_attitude', 'net_restrict_score', 'agea', 'eduyrs', 'stfeco']].describe().round(2))

print(f"\n Attitude by Economic Satisfaction Group ")
print(reg_df.groupby('econ_dissatisfied')['immig_attitude'].agg(['mean', 'std', 'count']).round(3))

 Descriptive Statistics: Country-Year Level (N=233) 
       mean_attitude  net_restrict_score  n_policy_actions  n_respondents
count         233.00              233.00            233.00         233.00
mean            5.12               -3.05             20.09        1686.12
std             0.77                8.54             13.50         586.69
min             2.87              -46.00              1.00         550.00
25%             4.61               -7.00             11.00        1384.00
50%             5.18               -3.00             16.00        1647.00
75%             5.71                1.00             27.00        1939.00
max             7.30               24.00             76.00        7651.00

 Descriptive Statistics: Individual Level (N=378,971)
       immig_attitude  net_restrict_score       agea     eduyrs     stfeco
count       378971.00           378971.00  378971.00  378971.00  378971.00
mean             5.09               -3.15      48.77      12.55       4.62
s

In [4]:
# Compute Pearson correlation between policy restrictiveness and attitudes for each country across its ESS waves

corr_results = []
for country in ess_merged['cntry'].unique():
    subset = ess_merged[ess_merged['cntry'] == country].dropna(
        subset=['immig_attitude', 'net_restrict_score']
    )
    if len(subset) >= 5:
        r, p = stats.pearsonr(subset['net_restrict_score'], subset['immig_attitude'])
        corr_results.append({
            'cntry': country,
            'r': round(r, 3),
            'p': round(p, 3),
            'n': len(subset)
        })

corr_df = pd.DataFrame(corr_results).sort_values('r')

# Romania has no policy variation so correlation is undefined — set to 0 (neutral)
if 'RO' not in corr_df['cntry'].values:
    corr_df = pd.concat([
        corr_df,
        pd.DataFrame([{'cntry': 'RO', 'r': 0.0, 'p': 1.0, 'n': 0}])
    ], ignore_index=True)

print("Country-level correlations: policy restrictiveness vs immigration attitude")
print(corr_df.to_string(index=False))

# Save for use in visualize notebook
corr_df.to_csv('country_correlations.csv', index=False)
print("\nSaved to country_correlations.csv")

Country-level correlations: policy restrictiveness vs immigration attitude
cntry      r     p     n
   LV -0.196 0.000  3377
   PT -0.162 0.000 16062
   LU -0.134 0.000  2188
   CY -0.110 0.000  5464
   NL -0.110 0.000 16703
   HU -0.088 0.000 15898
   DE -0.085 0.000 30868
   GB -0.085 0.000 18488
   HR -0.078 0.000  5682
   PL -0.050 0.000 17058
   SE -0.041 0.000 15975
   SI -0.032 0.000 12095
   FI -0.030 0.000 18765
   LT -0.017 0.075 10899
   FR -0.002 0.784 17069
   BE -0.002 0.818 15340
   DK  0.004 0.686 11540
   AT  0.006 0.482 13617
   IE  0.014 0.058 18954
   CH  0.014 0.123 13033
   ES  0.016 0.034 17345
   NO  0.019 0.024 14524
   EE  0.054 0.000 13774
   CZ  0.063 0.000 19059
   GR  0.066 0.000 11497
   BG  0.167 0.000 12016
   IT  0.185 0.000  9302
   SK  0.189 0.000 10682
   IS  0.201 0.000  3692
   RO    NaN   NaN  1899

Saved to country_correlations.csv


In [5]:
# OLS regression with country and year fixed effects
# Clustered standard errors by country

# Key variables:
# net_restrict_std — standardized policy restrictiveness (mean 0, SD 1)
# econ_dissatisfied — 1 if stfeco <= 4, 0 otherwise
# Interaction term tests whether policy effect differs by economic satisfaction
# Controls: age, years of education

m_fe = smf.ols(
    'immig_attitude ~ net_restrict_std * econ_dissatisfied + agea + eduyrs + C(cntry) + C(year)',
    data=reg_df
).fit(cov_type='cluster', cov_kwds={'groups': reg_df['cntry']})

# Extract only key variables — skip the country/year dummy coefficients
key_vars = [
    'net_restrict_std',
    'econ_dissatisfied',
    'net_restrict_std:econ_dissatisfied',
    'agea',
    'eduyrs'
]

results_df = pd.DataFrame(
    m_fe.summary().tables[1].data[1:],
    columns=m_fe.summary().tables[1].data[0]
)
results_df.columns = results_df.columns.str.strip()
results_df[''] = results_df[''].str.strip()
key_results = results_df[results_df[''].isin(key_vars)]

print("Fixed Effects Regression Results")
print("Outcome: Immigration attitude index (0-10)")
print("Controls: age, education | FE: country + year | SE: clustered by country")
print()
print(key_results.to_string(index=False))

# Save results for visualize notebook
key_results.to_csv('regression_results.csv', index=False)
print("\nSaved to regression_results.csv")

Fixed Effects Regression Results
Outcome: Immigration attitude index (0-10)
Controls: age, education | FE: country + year | SE: clustered by country

                                         coef   std err         z  P>|z|    [0.025    0.975]
                  net_restrict_std     0.0029     0.039     0.075  0.940    -0.073     0.078
                 econ_dissatisfied    -0.7199     0.040   -17.842  0.000    -0.799    -0.641
net_restrict_std:econ_dissatisfied     0.0189     0.035     0.547  0.584    -0.049     0.087
                              agea    -0.0046     0.001    -3.497  0.000    -0.007    -0.002
                            eduyrs     0.1204     0.008    15.410  0.000     0.105     0.136

Saved to regression_results.csv


In [6]:
#Repeat main regression using major-only policy score

reg_df_major = ess_merged.dropna(
    subset=['immig_attitude', 'net_major_score', 'agea', 'eduyrs', 'stfeco']
).copy()

reg_df_major['net_major_std'] = (
    (reg_df_major['net_major_score'] - reg_df_major['net_major_score'].mean()) /
    reg_df_major['net_major_score'].std()
)

m_major = smf.ols(
    'immig_attitude ~ net_major_std * econ_dissatisfied + agea + eduyrs + C(cntry) + C(year)',
    data=reg_df_major
).fit(cov_type='cluster', cov_kwds={'groups': reg_df_major['cntry']})

key_vars_major = [
    'net_major_std',
    'econ_dissatisfied',
    'net_major_std:econ_dissatisfied',
    'agea',
    'eduyrs'
]

results_major = pd.DataFrame(
    m_major.summary().tables[1].data[1:],
    columns=m_major.summary().tables[1].data[0]
)
results_major.columns = results_major.columns.str.strip()
results_major[''] = results_major[''].str.strip()
key_major = results_major[results_major[''].isin(key_vars_major)]

print(" Major Policy Changes Only ")
print("Outcome: Immigration attitude index (0-10)")
print("Controls: age, education | FE: country + year | SE: clustered by country")
print()
print(key_major.to_string(index=False))

key_major.to_csv('regression_results_major.csv', index=False)
print("\nSaved to regression_results_major.csv")

 Major Policy Changes Only 
Outcome: Immigration attitude index (0-10)
Controls: age, education | FE: country + year | SE: clustered by country

                                      coef   std err         z  P>|z|    [0.025    0.975]
                  net_major_std     0.0181     0.030     0.610  0.542    -0.040     0.076
              econ_dissatisfied    -0.7236     0.035   -20.747  0.000    -0.792    -0.655
net_major_std:econ_dissatisfied    -0.0941     0.025    -3.754  0.000    -0.143    -0.045
                           agea    -0.0046     0.001    -3.492  0.000    -0.007    -0.002
                         eduyrs     0.1203     0.008    15.459  0.000     0.105     0.136

Saved to regression_results_major.csv


In [7]:
reg_df_nonmajor = ess_merged.dropna(
    subset=['immig_attitude', 'net_nonmajor_score', 'agea', 'eduyrs', 'stfeco']
).copy()

reg_df_nonmajor['net_nonmajor_std'] = (
    (reg_df_nonmajor['net_nonmajor_score'] - reg_df_nonmajor['net_nonmajor_score'].mean()) /
    reg_df_nonmajor['net_nonmajor_score'].std()
)

m_nonmajor = smf.ols(
    'immig_attitude ~ net_nonmajor_std * econ_dissatisfied + agea + eduyrs + C(cntry) + C(year)',
    data=reg_df_nonmajor
).fit(cov_type='cluster', cov_kwds={'groups': reg_df_nonmajor['cntry']})

key_vars_nonmajor = [
    'net_nonmajor_std',
    'econ_dissatisfied',
    'net_nonmajor_std:econ_dissatisfied',
    'agea',
    'eduyrs'
]

results_nonmajor = pd.DataFrame(
    m_nonmajor.summary().tables[1].data[1:],
    columns=m_nonmajor.summary().tables[1].data[0]
)
results_nonmajor.columns = results_nonmajor.columns.str.strip()
results_nonmajor[''] = results_nonmajor[''].str.strip()
key_nonmajor = results_nonmajor[results_nonmajor[''].isin(key_vars_nonmajor)]

print("Sub-Analysis: Non-Major Policy Changes Only")
print("Captures fine-tuning, mid-level, and minor changes")
print("Outcome: Immigration attitude index (0-10)")
print("Controls: age, education | FE: country + year | SE: clustered by country")
print()
print(key_nonmajor.to_string(index=False))

key_nonmajor.to_csv('regression_results_nonmajor.csv', index=False)
print("\nSaved to regression_results_nonmajor.csv")

Sub-Analysis: Non-Major Policy Changes Only
Captures fine-tuning, mid-level, and minor changes
Outcome: Immigration attitude index (0-10)
Controls: age, education | FE: country + year | SE: clustered by country

                                         coef   std err         z  P>|z|    [0.025    0.975]
                  net_nonmajor_std    -0.0029     0.043    -0.068  0.946    -0.086     0.081
                 econ_dissatisfied    -0.7189     0.040   -18.051  0.000    -0.797    -0.641
net_nonmajor_std:econ_dissatisfied     0.0449     0.039     1.166  0.244    -0.031     0.120
                              agea    -0.0046     0.001    -3.496  0.000    -0.007    -0.002
                            eduyrs     0.1203     0.008    15.435  0.000     0.105     0.136

Saved to regression_results_nonmajor.csv


In [8]:
# Aggregate policy scores by wave for use in 03_visualize
major_time = ess_merged.groupby('year')['net_major_score'].mean().reset_index()
major_time.columns = ['year', 'net_major_score']
major_time.to_csv('major_policy_trend.csv', index=False)

nonmajor_time = ess_merged.groupby('year')['net_nonmajor_score'].mean().reset_index()
nonmajor_time.columns = ['year', 'net_nonmajor_score']
nonmajor_time.to_csv('nonmajor_policy_trend.csv', index=False)

print("Major policy trend by wave:")
print(major_time)
print("\nNon-major policy trend by wave:")
print(nonmajor_time)
print("\nSaved major_policy_trend.csv and nonmajor_policy_trend.csv")

Major policy trend by wave:
   year  net_major_score
0  2002        -1.282624
1  2004         0.207194
2  2006         0.538331
3  2008        -0.676065
4  2010        -0.547179
5  2012         0.120791
6  2014        -0.181667
7  2016         1.839199
8  2018         1.342307
9  2020         1.297986

Non-major policy trend by wave:
   year  net_nonmajor_score
0  2002           -0.268807
1  2004            1.312321
2  2006           -0.564144
3  2008           -4.069167
4  2010           -1.631503
5  2012           -0.837009
6  2014           -6.272887
7  2016           -9.426112
8  2018           -6.790059
9  2020           -5.754584

Saved major_policy_trend.csv and nonmajor_policy_trend.csv
